# Objectif
Notre objectif dans ce notebook est de transformer les données brut en PPT en une structure JSON bien claire et efficace à utiliser ultérieurement.

In [1]:
# structure finale voulue
# cette variable est à titre d'exemple, elle va s'écraser par la suite
course = {
    "course_name" : "Traitement du signal",
    "course_plan" : [
        {"chap_id" : 0, "chap_name" : "Introduction"},
        {"chap_id" : 1, "chap_name" : "Représentation et analyse temporelle des signaux"},
        {"chap_id" : 2, "chap_name" : "Systèmes linéaires continus"}
    ],
    "chapters" : [
        {
            "chap_id" : 0,
            "chap_name" : "Introduction",
            "content" : []
        },
        {
            "chap_id" : 1,
            "chap_name" : "Représentation temporelle des signaux",
            "content" : [
                {
                    "slide_id" : 23,
                    "section" : "Les différentes classes de signaux",
                    "categories" : {
                        "definition" : ["on definit .."],
                        "formula" : [
                                {
                                    "formula_id": 1,
                                    "type": "OMML",
                                    "extractable": True,
                                    "raw_text": "la formule ici"
                                }
                        ]
                    },
                    "notions" : ["Différents critères de classification"],
                    "slide_content" : 'Différents critères de classification :\n'
                                '\uf0a7 Nombre de variables ;\n'
                                '\uf0a7 Signaux réels / complexes ;\n'
                                '\uf0a7 Déterministes / aléatoires ;\n'
                                '\uf0a7 Continus / discrets ;\n'
                                '.\n'
                                '\uf0a7 Classification énergétique\n'
                                '23'
                },
                {
                    "slide_id" : 24,
                    "section" : "1- Classification en fonction du nombre de variables",
                    "categories" : {},
                    "notions" : [],
                    "slide_content" : 'Exemple de signal N. Variable (s) Type de signal'
                                'Tension électrique V(t) 1 monodimensionnel'
                                'Image f(x,y) 2 bidimensionnel'
                                'Vidéo I(x,y,t) 3 tridimensionnel'
                                '. . .'
                                '. . .'
                                '. . .'
                                'Onde e.m u(x,y,z,t) N multidimensionnel'
                                '24'
                }
            ]
        },
        {
        "chap_id" : 2,
        "chap_name" : "Systèmes linéaires continus\nDéfinition et caractérisation",
        "content" : []
        }
    ]
}

# lorsqu'on commence la partie category, on doit faire une modification sur la variable "course"
categories = ["definition", "exemple", "formula", "remark"]

# Initialisations
Pour simplifier, supposons maintenant que :
- le plan est fixé manuellement
- l'introduction est ignorée (l'introduction est le chapitre 0)
- **tous les fichiers `.pptx` du dossier `ppt` sont traités** : un document ppt = un chapitre (trié par nom de fichier)

In [2]:
from pprint import pprint
import os
import glob

# Dossier contenant les .pptx
PPT_DIR = "../data/1-ppt/"

# récupérer tous les vrais .pptx
PPT_FILES = sorted([
    f for f in glob.glob(os.path.join(PPT_DIR, "*.pptx"))
    if not os.path.basename(f).startswith("~$")
])

print(f"{len(PPT_FILES)} fichier(s) trouve(s) dans '{PPT_DIR}' :")

for i, path in enumerate(PPT_FILES, start=1):
    print(f"  Chapitre {i} -> {os.path.basename(path)}")

# Initialisation du cours
course = {}
course["course_name"] = "Traitement du signal"

course["course_plan"] = [
    {"chap_id": 0, "chap_name": "Introduction"}
]

course["chapters"] = []

print("\nInitialisation OK.")

2 fichier(s) trouve(s) dans '../data/1-ppt/' :
  Chapitre 1 -> Cours_TS_Chap1.pptx
  Chapitre 2 -> Cours_TS_Chap2.pptx

Initialisation OK.


# Extraction du nom de chapitre
Pour structurer le chapitre 1 (= le dictionnaire course["chapters"][0]), on commence par extraire le **nom du chapitre** (extrait du slide 1 du ppt) :

In [4]:
import os
from pptx import Presentation

def extract_chapter_name(first_slide_text, ppt_path=""):
    lines = first_slide_text.split("\n")
    # Nettoyer lignes vides
    lines = [line.strip() for line in lines if line.strip()]
    # Enlever "Chapitre ..."
    if lines and "chapitre" in lines[0].lower():
        lines = lines[1:]
    # Enlever numero de slide (ex: "22")
    if lines and lines[-1].isdigit():
        lines = lines[:-1]
    # Prendre premiere ligne
    if lines:
        return lines[0]
    # Fallback -> nom du fichier ppt
    return os.path.basename(ppt_path).replace(".pptx", "")
    
# ── Parcours des chapitres ─────────────────────────────
course["chapters"]=[]
for i, file in enumerate(PPT_FILES, start=1):
    prs = Presentation(file)

    # première slide
    first_slide = prs.slides[0]

    # texte de la slide
    first_slide_text = "\n".join(
        shape.text
        for shape in first_slide.shapes
        if shape.has_text_frame
    )

    # nom du chapitre
    chapter_name = extract_chapter_name(first_slide_text, file)

    # sauvegarde
    chapter = {
        "chap_id": i,
        "chap_name": chapter_name,
        "content": []
        
    }

    course["chapters"].append(chapter)

    # affichage
    print(f"Chapitre {i} : {chapter_name}")


# ── Résultat final ────────────────────────────────────

print("\nListe des chapitres :")
pprint(course["chapters"])

Chapitre 1 : Représentation temporelle des signaux
Chapitre 2 : Systèmes linéaires continus

Liste des chapitres :
[{'chap_id': 1,
  'chap_name': 'Représentation temporelle des signaux',
  'content': []},
 {'chap_id': 2, 'chap_name': 'Systèmes linéaires continus', 'content': []}]


# Extraction de la section d'un slide
On passe au contenu du chapitre, qui est une liste de slides.  
On commence par extraire le **titre/section** de chaque slide.  

*Remarque :* L’extraction des sections a posé des difficultés en raison de titres non correctement détectés. Une **étape manuelle** a donc été nécessaire, consistant à ajuster le layout des slides dans PowerPoint afin de définir explicitement les titres.

In [5]:
import re
from pptx import Presentation

def clean_section_title(title):
    """
    Nettoie le titre :
    - supprime les chiffres au début
    - supprime les tirets/points après le numéro
    - supprime les espaces inutiles
    """

    if not title:
        return ""

    # enlever espaces début/fin
    title = title.strip()

    # supprimer : 1-  / 2. / 3 ) / 4 :
    title = re.sub(r'^\d+\s*[-.):]*\s*', '', title)

    # supprimer espaces multiples
    title = re.sub(r'\s+', ' ', title)

    return title.strip()


def extract_slide_section(slide):

    # récupérer le vrai titre du slide
    if slide.shapes.title and slide.shapes.title.text.strip():
        section = slide.shapes.title.text.strip()

    else:
        # fallback → première ligne texte trouvée
        all_text = []

        for shape in slide.shapes:
            if shape.has_text_frame:
                for p in shape.text_frame.paragraphs:
                    txt = p.text.strip()

                    if txt:
                        all_text.append(txt)

        section = all_text[0] if all_text else ""

    # nettoyage
    section = clean_section_title(section)

    return section


for file in PPT_FILES:

    prs = Presentation(file)

    print(f"\n========== {file} ==========\n")

    for i, slide in enumerate(prs.slides):

        section = extract_slide_section(slide)

        print(f"Slide {i+1} section: {section}")
        print("------")


========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Slide 1 section: Chapitre 1
------
Slide 2 section: Les différentes classes de signaux
------
Slide 3 section: Classification en fonction du nombre de variables
------
Slide 4 section: Signaux réels / complexes
------
Slide 5 section: Signaux déterministes / aléatoires
------
Slide 6 section: Signaux déterministes / aléatoires
------
Slide 7 section: Signaux déterministes / aléatoires
------
Slide 8 section: Signaux continus / discrets
------
Slide 9 section: Signaux continus / discrets
------
Slide 10 section: Signaux continus / discrets
------
Slide 11 section: Signaux continus / discrets
------
Slide 12 section: Signaux continus / discrets
------
Slide 13 section: Signaux continus / discrets
------
Slide 14 section: Classification énergétique
------
Slide 15 section: Classification énergétique
------
Slide 16 section: Classification énergétique
------
Slide 17 section: Classification énergétique
------
Slide 18 section: Class

# Extraction du contenu d'un slide
Puis on extrait le **contenu du slide** :

In [6]:
from pptx import Presentation
from pptx.enum.shapes import PP_PLACEHOLDER
from lxml import etree

MC_NS   = "http://schemas.openxmlformats.org/markup-compatibility/2006"
DML_NS  = "http://schemas.openxmlformats.org/drawingml/2006/main"
MATH_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"

# ── Équations OOXML ──────────────────────────────────────────────────────────

def extract_math(omath_el):
    tokens = []

    def walk(el):
        tag = etree.QName(el.tag).localname
        if tag == 't':
            if el.text: tokens.append(el.text)
        elif tag == 'f':
            num = el.find(f'{{{MATH_NS}}}num')
            den = el.find(f'{{{MATH_NS}}}den')
            n = "".join(t.text or "" for t in num.iter(f'{{{MATH_NS}}}t')) if num is not None else ""
            d = "".join(t.text or "" for t in den.iter(f'{{{MATH_NS}}}t')) if den is not None else ""
            tokens.append(f"({n}/{d})")
        elif tag == 'sSup':
            base = el.find(f'{{{MATH_NS}}}e')
            sup  = el.find(f'{{{MATH_NS}}}sup')
            b = "".join(t.text or "" for t in base.iter(f'{{{MATH_NS}}}t')) if base is not None else ""
            s = "".join(t.text or "" for t in sup.iter(f'{{{MATH_NS}}}t'))  if sup  is not None else ""
            tokens.append(f"{b}^{s}")
        elif tag == 'sSub':
            base = el.find(f'{{{MATH_NS}}}e')
            sub  = el.find(f'{{{MATH_NS}}}sub')
            b = "".join(t.text or "" for t in base.iter(f'{{{MATH_NS}}}t')) if base is not None else ""
            s = "".join(t.text or "" for t in sub.iter(f'{{{MATH_NS}}}t'))  if sub  is not None else ""
            tokens.append(f"{b}_{s}")
        elif tag == 'rad':
            deg  = el.find(f'{{{MATH_NS}}}deg')
            base = el.find(f'{{{MATH_NS}}}e')
            d = "".join(t.text or "" for t in deg.iter(f'{{{MATH_NS}}}t'))  if deg  is not None else ""
            b = "".join(t.text or "" for t in base.iter(f'{{{MATH_NS}}}t')) if base is not None else ""
            tokens.append(f"√{d}({b})" if d.strip() else f"√({b})")
        elif tag == 'd':
            tokens.append("(")
            for child in el:
                if etree.QName(child.tag).localname == 'e': walk(child)
            tokens.append(")")
        elif tag not in ('rPr','mPr','dPr','fPr','sSupPr','sSubPr',
                         'radPr','ctrlPr','argPr','groupChrPr'):
            for child in el: walk(child)

    walk(omath_el)
    return "".join(tokens)


# ── Paragraphe complet (runs + champs + équations) ───────────────────────────

def extract_paragraph_full(para_el, slide_number=None):
    parts = []
    for child in para_el:
        local = etree.QName(child.tag).localname
        if local == 'r':
            t = child.find(f'{{{DML_NS}}}t')
            if t is not None and t.text:
                parts.append(t.text)
        elif local == 'fld':
            fld_type = child.get('type', '')
            t = child.find(f'{{{DML_NS}}}t')
            if fld_type == 'slidenum':
                parts.append(str(slide_number) if slide_number is not None else '[SLIDE_NUM]')
            elif fld_type == 'datetime':
                parts.append(t.text if t is not None and t.text else '[DATE]')
            else:
                parts.append(t.text if t is not None and t.text else f'[FLD:{fld_type}]')
        elif local == 'm':
            omath = child.find(f'{{{MATH_NS}}}oMath')
            if omath is not None:
                parts.append(f"[EQ: {extract_math(omath)}]")
    return "".join(parts)
    
def extract_slide_content(slide, section, slide_number):
    seen   = set()
    blocks = []

    def add(text):
        text = text.strip()
        if (text
                and text != section
                and text not in seen
                and text != str(slide_number)):   # ← exclure le numéro de slide
            seen.add(text)
            blocks.append(text)

    # 1. Shapes normales
    for shape in slide.shapes:
        if not shape.has_text_frame:
            continue
        if shape.is_placeholder:
            ph_type = shape.placeholder_format.type
            # Ignorer titre ET numéro de slide
            if ph_type in {PP_PLACEHOLDER.TITLE,
                           PP_PLACEHOLDER.CENTER_TITLE,
                           PP_PLACEHOLDER.SLIDE_NUMBER}:   # ← ignorer le placeholder entier
                continue

        for para in shape.text_frame._txBody.findall(f'{{{DML_NS}}}p'):
            line = extract_paragraph_full(para, slide_number)
            add(line)

    # 2. AlternateContent
    for child in slide.shapes._spTree:
        if child.tag != f'{{{MC_NS}}}AlternateContent':
            continue
        choice = child.find(f'{{{MC_NS}}}Choice')
        source = choice if choice is not None else child.find(f'{{{MC_NS}}}Fallback')
        if source is None:
            continue
        for para in source.findall(f'.//{{{DML_NS}}}p'):
            line = extract_paragraph_full(para, slide_number)
            add(line)

    return "\n".join(blocks)


# ── Point d'entrée ───────────────────────────────────────────────────────────

for i, file in enumerate(PPT_FILES, start=1):
    prs = Presentation(file)
    print(f"\n========== {file} ==========\n")
    for i, slide in enumerate(prs.slides):
        slide_number = i + 1
        section      = extract_slide_section(slide)
        content      = extract_slide_content(slide, section, slide_number)
    
        print(f"Slide {slide_number} — section : {section}")
        print("------")
        if content:                          # ← n'afficher que si non vide
            print(f"Content:\n{content}")
            print("======")


========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Slide 1 — section : Chapitre 1
------
Content:
Représentation temporelle des signaux
Slide 2 — section : Les différentes classes de signaux
------
Content:
Différents critères de classification :
Nombre de variables ;
Signaux réels / complexes ;
Déterministes / aléatoires ;
Continus / discrets ;
Classification énergétique.
Slide 3 — section : Classification en fonction du nombre de variables
------
Slide 4 — section : Signaux réels / complexes
------
Content:
signal réel  :    si   [EQ: 𝑥(𝑡)∈ℝ]
signal complexe : si   [EQ: 𝑥(𝑡)∈ℂ]
Notations usuelles:
Nombre complexe  [EQ: 𝑗: 𝑗^2=−1]
Si [EQ: 𝑥(𝑡)=𝑎(𝑡)+𝑗 𝑏(𝑡)]  :
partie réelle : [EQ: 𝑅𝑒(𝑥(𝑡))=𝑎(𝑡)]
partie imaginaire:  [EQ: 𝐼𝑚(𝑥(𝑡))=𝑏(𝑡)]
module: [EQ: (𝑥(𝑡))=√(𝑎2𝑡+𝑏2𝑡)]
argument: [EQ: 𝐴𝑟𝑔(𝑥(𝑡))=𝑎𝑟𝑐𝑡𝑎𝑛((𝑏(𝑡)/𝑎(𝑡)))]
[EQ: 𝑥(𝑡)=(𝑥(𝑡))𝑒^𝑗𝐴𝑟𝑔𝑥𝑡]
Complexe conjugué:   [EQ: 𝑥^∗(𝑡)=𝑎(𝑡)−𝑗 𝑏(𝑡)=(𝑥(𝑡))𝑒^−𝑗𝐴𝑟𝑔𝑥𝑡]
Slide 5 — section : Signaux déterministes / aléatoires
------
Content:
Signaux dét

# Extraction des notions d'un slide
Puis on extrait les **notions du slide** :  
Les candidats sont extraits via spaCy : le texte du slide est analysé grammaticalement et seuls les groupes nominaux sont retenus comme candidats potentiels de notions. Cela garantit que les candidats sont des unités linguistiques cohérentes (ex: "signaux périodiques", "énergie finie") plutôt que des fragments arbitraires.

In [7]:
import spacy
import re
from pptx import Presentation

nlp = spacy.load("fr_core_news_sm")


def clean_text(text):
    """
    Nettoyage général du texte
    """

    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()


def remove_semantic_duplicates(candidates):
    """
    Supprime les notions courtes incluses dans des notions plus longues

    Exemple :
        "signaux"
        "signaux déterministes"

    => "signaux" supprimé
    """

    # Trier par longueur décroissante
    sorted_candidates = sorted(candidates, key=len, reverse=True)

    final = []

    for cand in sorted_candidates:

        cand_lower = cand.lower()

        # vérifier si déjà inclus dans une notion plus longue
        included = False

        for kept in final:
            kept_lower = kept.lower()

            # mot entier inclus
            if re.search(rf"\b{re.escape(cand_lower)}\b", kept_lower):
                included = True
                break

        if not included:
            final.append(cand)

    # remettre ordre naturel
    final.reverse()

    return final


def extract_candidates(slide_text, section_title=None, min_len=3):

    doc = nlp(slide_text)

    seen = set()
    candidates = []

    for chunk in doc.noun_chunks:

        tokens = []

        for token in chunk:

            # ponctuation
            if token.is_punct:
                continue

            # stop words spaCy
            if token.is_stop:
                continue

            # déterminants / prépositions
            if token.pos_ in {"DET", "ADP"}:
                continue

            # mots trop courts
            if len(token.text.strip()) < min_len:
                continue

            tokens.append(token.text)

        cleaned = " ".join(tokens)

        cleaned = clean_text(cleaned)

        if not cleaned:
            continue

        # longueur minimale
        if len(cleaned) < min_len:
            continue

        # uniquement chiffres
        if cleaned.isdigit():
            continue

        text_norm = cleaned.lower()

        # liens
        if re.search(r"https?://|www\.", text_norm):
            continue

        # contient chiffres
        if re.search(r"[0-9]", cleaned):
            continue

        # caractères spéciaux
        if re.search(r"[^a-zA-ZÀ-ÿ\s\-]", cleaned):
            continue

        # antislash
        if "\\" in cleaned:
            continue

        # caractères de contrôle
        if re.search(r"[\x00-\x1F\x7F]", cleaned):
            continue

        # doublons exacts
        if text_norm in seen:
            continue

        seen.add(text_norm)

        candidates.append(cleaned)

    # suppression des doublons sémantiques
    candidates = remove_semantic_duplicates(candidates)

    # fallback :
    # si aucune notion trouvée → utiliser le titre du slide
    if not candidates and section_title:

        section_title = clean_text(section_title)

        if len(section_title) >= min_len:
            candidates = [section_title]

    return candidates


# -------- lecture PowerPoint --------

for file in PPT_FILES:

    prs = Presentation(file)

    print(f"\n========== {file} ==========\n")

    for i, slide in enumerate(prs.slides):

        slide_number = i + 1

        section = extract_slide_section(slide)

        content = extract_slide_content(
            slide,
            section,
            slide_number
        )

        notions = extract_candidates(
            content,
            section_title=section
        )

        print(f"Slide {slide_number} notions: {notions}")
        print("------")


========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Slide 1 notions: ['signaux', 'Représentation temporelle']
------
Slide 2 notions: ['Nombre', 'discrets', 'Continus', 'critères', 'variables', 'aléatoires', 'Déterministes', 'Signaux réels complexes', 'Classification énergétique']
------
Slide 3 notions: ['Classification en fonction du nombre de variables']
------
Slide 4 notions: ['argument', 'partie réelle', 'Nombre complexe', 'Complexe conjugué', 'partie imaginaire']
------
Slide 5 notions: ['temps', 'classe', 'signal', 'valeur', 'instant', 'évolution', 'signaux périodiques', 'fonction mathématique', 'Signaux déterministes']
------
Slide 6 notions: ['loi', 'temps', 'appel', 'bruit', 'moyenne', 'décrire', 'Exemple', 'variance', 'évolution', 'expérience', 'probabilité', 'Signaux aléatoires', 'propriétés statistiques']
------
Slide 7 notions: ['bruit', 'signal']
------
Slide 8 notions: ['heure', 'cours', 'types', 'bourse', 'instant', 'variable', 'instants Exemple', 'signal analog

On remarque que les notions associées à chaque slide ne sont pas convenables. En effet, même si le contenu est traité lignuistiquement et structuré par groupes nominaux, on n'a pas une façon réelle pour différencier entre une notion(ex: "signaux périodiques", "énergie finie") et un mot quelconque(ex : "Nombre", etc).  
**On doit alors ajouter à cette méthode de structuration grammaticale une méthode de compréhension sémantique.**

La méthode de compréhension sémantique qu'on utilisera est l'**embeddings**, consistant à transformer les mots en vecteurs numériques denses permettant aux machines de comprendre le sens des mots et leur proximité **sémantique**.  
On commence par importer le modèle responsable de la représentation vectorielle du texte.

In [8]:
import os
os.environ["TQDM_DISABLE"] = "1"

# Force tqdm to use plain text mode - must happen before sentence_transformers import
from tqdm.auto import tqdm as tqdm_auto
from tqdm import tqdm as tqdm_std
import tqdm.auto
tqdm.auto.tqdm = tqdm_std  # replace notebook/widget tqdm with plain text tqdm

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1533.95it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Voici la fonction qui extrait les notions de chaque slide en combinant une analyse linguistique et une méthode sémantique par embeddings.  

Son principe consiste à d'abord générer des candidats linguistiquement valides à partir du contenu du slide, puis à sélectionner parmi eux les plus pertinents par rapport au contexte du slide.  

Pour la similarité sémantique, la référence utilisée pour la comparaison est construite à partir de trois niveaux de contexte : le nom du chapitre, le titre de la section, et une partie du contenu du slide.  

Les étapes sont les suivantes :
- analyser le texte du slide et extraire les groupes nominaux comme candidats
- construire la référence contextuelle : `chapitre + section + contenu (50 premiers mots)`
- calculer l'embedding de la référence et l'embedding de chaque candidat
- comparer les embeddings à l'aide de la similarité cosinus (valeur entre 0 et 1, où 1 indique une forte similarité)
- trier les candidats par ordre décroissant de similarité
- sélectionner les `top_k` candidats dépassant le seuil `threshold` comme notions

In [9]:
def extract_notions(
    chapter_name,
    section_title,
    slide_text,
    top_k=3,
    threshold=0.35
):
    """
    Extraction des notions pertinentes d'un slide
    avec :
      - nettoyage
      - suppression stop words
      - suppression mots courts
      - suppression doublons sémantiques
      - fallback sur le titre du slide
    """

    # -------- extraction des candidats --------
    candidates = extract_candidates(
        slide_text,
        section_title=section_title,
        min_len=3
    )

    # fallback sécurité
    if not candidates:

        cleaned_title = clean_text(section_title)

        if cleaned_title:
            return [cleaned_title]

        return []

    # -------- contexte multi-niveaux --------

    short_content = " ".join(slide_text.split()[:50])

    query = (
        f"query: "
        f"{chapter_name}. "
        f"{section_title}. "
        f"{short_content}"
    )

    passages = [
        f"passage: {c}"
        for c in candidates
    ]

    # -------- embeddings --------

    query_emb = model.encode(
        [query],
        normalize_embeddings=True
    )

    passage_embs = model.encode(
        passages,
        normalize_embeddings=True
    )

    # -------- similarité cosinus --------

    scores = np.dot(
        passage_embs,
        query_emb.T
    ).flatten()

    # -------- ranking --------

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # -------- filtrage --------

    notions = [
        text
        for text, score in ranked
        if score >= threshold
    ]

    # -------- suppression doublons sémantiques --------

    notions = remove_semantic_duplicates(notions)

    # -------- top_k --------

    notions = notions[:top_k]

    # -------- fallback final --------

    if not notions:

        cleaned_title = clean_text(section_title)

        if cleaned_title:
            notions = [cleaned_title]

    return notions


# -------- lecture des fichiers PowerPoint --------

for file in PPT_FILES:

    prs = Presentation(file)

    # -------- nom du chapitre --------

    first_slide_text = "\n".join(
        shape.text
        for shape in prs.slides[0].shapes
        if shape.has_text_frame
    )

    chap_name = extract_chapter_name(
        first_slide_text,
        file
    )

    print(f"\n========== {file} ==========\n")

    # -------- parcours slides --------

    for i, slide in enumerate(prs.slides):

        slide_number = i + 1

        section = extract_slide_section(slide)

        content = extract_slide_content(
            slide,
            section,
            slide_number
        )

        notions = extract_notions(
            chap_name,
            section,
            content,
            top_k=3,
            threshold=0.35
        )

        print(f"Slide {slide_number} notions: {notions}")
        print("------")


========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Slide 1 notions: ['signaux', 'Représentation temporelle']
------
Slide 2 notions: ['Nombre', 'Continus', 'discrets']
------
Slide 3 notions: ['Classification en fonction du nombre de variables']
------
Slide 4 notions: ['argument', 'partie réelle', 'Nombre complexe']
------
Slide 5 notions: ['temps', 'classe', 'signal']
------
Slide 6 notions: ['loi', 'appel', 'temps']
------
Slide 7 notions: ['bruit', 'signal']
------
Slide 8 notions: ['types', 'cours', 'heure']
------
Slide 9 notions: ['types', 'signaux temporels']
------
Slide 10 notions: ['Signaux continus / discrets']
------
Slide 11 notions: ['Signaux continus / discrets']
------
Slide 12 notions: ['Signaux continus / discrets']
------
Slide 13 notions: ['temps', 'basses', 'signal']
------
Slide 14 notions: ['signal continu', 'Puissance instantanée']
------
Slide 15 notions: ['somme', 'puissance', 'Energie totale']
------
Slide 16 notions: ['signal', 'valeur moyenne', 'Pui

Le résultat donné par ces deux méthodes combinés est plus efficace que la 1ère méthode, toujours avec une marge d'amélioration possible.

# Extraction des catégories dans un slide
On passe à **catégoriser le contenu du slide**. Pour l'instant, les catégories possibles du contenu d'un slide qu'on va fixer sont : définition, exemple, remarque, formule mathématique. Un slide peut alors avoir plus qu'une seule catégorie.  
On commence par détecter les formules :

In [10]:
from pptx import Presentation
from lxml import etree

MC_NS   = "http://schemas.openxmlformats.org/markup-compatibility/2006"
DML_NS  = "http://schemas.openxmlformats.org/drawingml/2006/main"
MATH_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"


# ── Détection et extraction des formules ─────────────────────────────────────

def detect_formulas(slide, slide_number):
    """
    Détecte et extrait toutes les formules mathématiques d'un slide.

    Types gérés :
      - OMML  (<m:oMath>)  : format moderne, texte extrait via extract_math()
      - OLE   (shape_type 7) : ancien Equation Editor 3.0, non extractable
    
    Retourne une liste de dicts :
      {
        formula_id   : int,
        slide        : int,
        shape_name   : str,
        type         : "OMML" | "OLE",
        extractable  : bool,
        text         : str | None,   # résultat lisible ou None
        raw_itertext : str | None,   # texte brut XML (debug)
      }
    """
    formulas   = []
    formula_id = 1

    # ── 1. Shapes normales ───────────────────────────────────────────────────
    for shape in slide.shapes:
        xml_str = shape._element.xml

        # 1a. OMML dans une shape normale
        if "<m:oMath" in xml_str:
            tree          = etree.fromstring(xml_str.encode())
            math_elements = tree.findall(f".//{{{MATH_NS}}}oMath")
            for m in math_elements:
                readable   = extract_math(m).strip()
                raw_itext  = "".join(m.itertext()).strip()
                formulas.append({
                    "formula_id"   : formula_id,
                    "slide"        : slide_number,
                    "shape_name"   : shape.name,
                    "type"         : "OMML",
                    "extractable"  : True,
                    "text"         : readable  or raw_itext or None,
                    "raw_itertext" : raw_itext or None,
                })
                formula_id += 1

        # 1b. OLE — ancien Equation Editor
        elif shape.shape_type == 7:
            formulas.append({
                "formula_id"   : formula_id,
                "slide"        : slide_number,
                "shape_name"   : shape.name,
                "type"         : "OLE",
                "extractable"  : False,
                "text"         : None,
                "raw_itertext" : None,
            })
            formula_id += 1

    # ── 2. AlternateContent (équations embarquées dans du texte) ─────────────
    for child in slide.shapes._spTree:
        if child.tag != f'{{{MC_NS}}}AlternateContent':
            continue

        choice = child.find(f'{{{MC_NS}}}Choice')
        source = choice if choice is not None else child.find(f'{{{MC_NS}}}Fallback')
        if source is None:
            continue

        math_elements = source.findall(f".//{{{MATH_NS}}}oMath")
        for m in math_elements:
            readable  = extract_math(m).strip()
            raw_itext = "".join(m.itertext()).strip()
            formulas.append({
                "formula_id"   : formula_id,
                "slide"        : slide_number,
                "shape_name"   : "AlternateContent",
                "type"         : "OMML",
                "extractable"  : True,
                "text"         : readable  or raw_itext or None,
                "raw_itertext" : raw_itext or None,
            })
            formula_id += 1

    return formulas


# ── Extraction globale ────────────────────────────────────────────────────────

def extract_all_formulas(prs):
    all_formulas = []

    for i, slide in enumerate(prs.slides):
        slide_formulas = detect_formulas(slide, slide_number=i + 1)
        all_formulas.extend(slide_formulas)

    # Garder uniquement :
    #  - les OLE (non extractables, on ne peut pas vérifier)
    #  - les OMML dont le texte contient "="
    all_formulas = [
        f for f in all_formulas
        if not f["extractable"] or (f["text"] and "=" in f["text"])
    ]

    # -------- renumérotation globale à partir de 1 --------
    for idx, f in enumerate(all_formulas, start=1):
        f["formula_id"] = idx

    stats = {
        "total"      : len(all_formulas),
        "omml"       : sum(1 for f in all_formulas if f["type"] == "OMML"),
        "ole"        : sum(1 for f in all_formulas if f["type"] == "OLE"),
        "extractable": sum(1 for f in all_formulas if f["extractable"]),
        "per_slide"  : {},
    }
    for f in all_formulas:
        stats["per_slide"].setdefault(f["slide"], 0)
        stats["per_slide"][f["slide"]] += 1

    return all_formulas, stats


# ── Affichage ─────────────────────────────────────────────────────────────────

def print_formulas(all_formulas, stats):
    print(f"\n{'═'*55}")
    print(f"  {stats['total']} formule(s) — "
          f"{stats['omml']} OMML / {stats['ole']} OLE / "
          f"{stats['extractable']} extractable(s)")
    print(f"{'═'*55}")

    current_slide = None
    for f in all_formulas:
        if f["slide"] != current_slide:
            current_slide = f["slide"]
            print(f"\n── Slide {current_slide} "
                  f"({stats['per_slide'][current_slide]} formule(s)) ──")

        status = "✓" if f["extractable"] else "✗"
        print(f"  [{status}] #{f['formula_id']}  {f['type']}"
              f"  shape=«{f['shape_name']}»")
        if f["text"]:
            print(f"       → {f['text']}")
        else:
            print(f"       → (non extractable)")

    print(f"\n{'═'*55}\n")


# ── Point d'entrée ────────────────────────────────────────────────────────────

for file in PPT_FILES:

    prs = Presentation(file)

    # Récupération du nom du chapitre depuis la première slide
    first_slide_text = "\n".join(
        shape.text
        for shape in prs.slides[0].shapes
        if shape.has_text_frame
    )

    chap_name = extract_chapter_name(first_slide_text, file)

    print(f"\n========== {file} ==========\n")
    print(f"Chapitre : {chap_name}\n")

    # Extraction des formules DU fichier courant
    all_formulas, stats = extract_all_formulas(prs)

    # Affichage
    print_formulas(all_formulas, stats)



========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Chapitre : Représentation temporelle des signaux


═══════════════════════════════════════════════════════
  38 formule(s) — 38 OMML / 0 OLE / 38 extractable(s)
═══════════════════════════════════════════════════════

── Slide 4 (8 formule(s)) ──
  [✓] #1  OMML  shape=«AlternateContent»
       → 𝑗: 𝑗^2=−1
  [✓] #2  OMML  shape=«AlternateContent»
       → 𝑥(𝑡)=𝑎(𝑡)+𝑗 𝑏(𝑡)
  [✓] #3  OMML  shape=«AlternateContent»
       → 𝑅𝑒(𝑥(𝑡))=𝑎(𝑡)
  [✓] #4  OMML  shape=«AlternateContent»
       → 𝐼𝑚(𝑥(𝑡))=𝑏(𝑡)
  [✓] #5  OMML  shape=«AlternateContent»
       → (𝑥(𝑡))=√(𝑎2𝑡+𝑏2𝑡)
  [✓] #6  OMML  shape=«AlternateContent»
       → 𝐴𝑟𝑔(𝑥(𝑡))=𝑎𝑟𝑐𝑡𝑎𝑛((𝑏(𝑡)/𝑎(𝑡)))
  [✓] #7  OMML  shape=«AlternateContent»
       → 𝑥(𝑡)=(𝑥(𝑡))𝑒^𝑗𝐴𝑟𝑔𝑥𝑡
  [✓] #8  OMML  shape=«AlternateContent»
       → 𝑥^∗(𝑡)=𝑎(𝑡)−𝑗 𝑏(𝑡)=(𝑥(𝑡))𝑒^−𝑗𝐴𝑟𝑔𝑥𝑡

── Slide 15 (1 formule(s)) ──
  [✓] #9  OMML  shape=«AlternateContent»
       → 𝐸_𝑥=−∞+∞𝑥(𝑡)^2𝑑𝑡

── Slide 16 (1 formule(s)) ──
  [✓] #1

In [11]:
import re
import os

IMAGE_OUTPUT_DIR = "../data/extracted_images"

CATEGORY_KEYWORDS = {
    "definition": [
        r"\bon appelle\b", r"\bon definit\b", r"\bs\'appelle\b",
        r"\best defini\b", r"\best appele\b", r"\bdefinition\b",
        r"\bon note\b", r"\bse definit\b", r"\bcorrespond a\b",
        r"\bse note\b", r"\bdesigne\b"
    ],
    "exemple": [
        r"\bexemple\b", r"\bpar exemple\b", r"\billustration\b",
        r"\bapplication\b", r"\bcas particulier\b", r"\bnotamment\b",
        r"\be\.g\b"
    ],
    "remarque": [
        r"\bremarque\b", r"\bnote\b", r"\battention\b",
        r"\bon remarque\b", r"\bon note que\b", r"\bobservation\b",
        r"\ben pratique\b", r"\ben effet\b"
    ]
}

def get_slide_image_paths(slide_number, chap_id=1, image_dir=IMAGE_OUTPUT_DIR):
    """
    Retourne la liste des chemins des images extraites pour un slide donné.
    Convention de nommage : data/extracted_images/chapitre{C}/slide{N}_img{M}.ext
    """
    # Cherche dans le sous-dossier chapitre{chap_id}
    chap_dir = os.path.join(image_dir, f"chapitre{chap_id}")
    if not os.path.isdir(chap_dir):
        return []
    prefix = f"slide{slide_number}_img"
    return sorted(
        os.path.join(chap_dir, fname)
        for fname in os.listdir(chap_dir)
        if fname.startswith(prefix)
    )

def extract_categories(slide, slide_content, slide_number, chap_id=1):
    # Détection des formules
    formulas = detect_formulas(slide, slide_number)
    formulas = [
        f for f in formulas
        if not f["extractable"] or (f["text"] and "=" in f["text"])
    ]

    # Chemins des images pour ce slide (chap_id transmis correctement)
    image_paths = get_slide_image_paths(slide_number, chap_id)

    categories = {
        "definition" : [],
        "formule"    : formulas,
        "exemple"    : [],
        "remarque"   : [],
        "image"      : image_paths,   # images associées au bon slide
    }

    lines = slide_content.split("\n")
    for line in lines:
        line_lower = line.lower().strip()
        if not line_lower:
            continue
        for category, patterns in CATEGORY_KEYWORDS.items():
            for pattern in patterns:
                if re.search(pattern, line_lower):
                    if category in categories:
                        categories[category].append(line.strip())
                    break

    categories = {k: v for k, v in categories.items() if v}
    return categories



for file in PPT_FILES:

    prs = Presentation(file)

    # Récupération du nom du chapitre depuis la première slide
    first_slide_text = "\n".join(
        shape.text
        for shape in prs.slides[0].shapes
        if shape.has_text_frame
    )

    chap_name = extract_chapter_name(first_slide_text, file)

    print(f"\n========== {file} ==========\n")
    print(f"Chapitre : {chap_name}\n")
    for i, slide in enumerate(prs.slides):
        slide_number = i + 1
        section    = extract_slide_section(slide)
        content    = extract_slide_content(slide, section, slide_number)
        categories = extract_categories(slide, content, slide_number, chap_id=1)
        print(f"Slide {slide_number} categories: {categories}")
        print("------")



========== ../data/1-ppt\Cours_TS_Chap1.pptx ==========

Chapitre : Représentation temporelle des signaux

Slide 1 categories: {}
------
Slide 2 categories: {}
------
Slide 3 categories: {'image': ['../data/extracted_images\\chapitre1\\slide3_img1.png', '../data/extracted_images\\chapitre1\\slide3_img2.png', '../data/extracted_images\\chapitre1\\slide3_img3.png']}
------
Slide 4 categories: {'formule': [{'formula_id': 3, 'slide': 4, 'shape_name': 'AlternateContent', 'type': 'OMML', 'extractable': True, 'text': '𝑗:\xa0𝑗^2=−1', 'raw_itertext': '𝑗:\xa0𝑗2=−1'}, {'formula_id': 4, 'slide': 4, 'shape_name': 'AlternateContent', 'type': 'OMML', 'extractable': True, 'text': '𝑥(𝑡)=𝑎(𝑡)+𝑗\xa0𝑏(𝑡)', 'raw_itertext': '𝑥𝑡=𝑎𝑡+𝑗\xa0𝑏𝑡'}, {'formula_id': 5, 'slide': 4, 'shape_name': 'AlternateContent', 'type': 'OMML', 'extractable': True, 'text': '𝑅𝑒(𝑥(𝑡))=𝑎(𝑡)', 'raw_itertext': '𝑅𝑒𝑥𝑡=𝑎𝑡'}, {'formula_id': 6, 'slide': 4, 'shape_name': 'AlternateContent', 'type': 'OMML', 'extractable': True, 'text': '𝐼𝑚(𝑥(

# Extraction des images 
La partie suivante nous permet de  extraire les images du cours .

In [12]:
from pptx import Presentation
import os

# Dossier racine des images
output_dir = "../data/extracted_images"
os.makedirs(output_dir, exist_ok=True)

total_images = 0

for chap_idx, ppt_path in enumerate(PPT_FILES, start=1):

    prs = Presentation(ppt_path)
    image_count = 0

    # Sous-dossier du chapitre
    chap_dir = os.path.join(output_dir, f"chapitre{chap_idx}")
    os.makedirs(chap_dir, exist_ok=True)

    for slide_index, slide in enumerate(prs.slides):

        for shape in slide.shapes:

            # 13 = Picture
            if shape.shape_type == 13:

                image = shape.image
                image_bytes = image.blob
                ext = "." + image.ext

                image_count += 1

                # Nom de l'image
                image_filename = (
                    f"slide{slide_index+1}_img{image_count}{ext}"
                )

                image_path = os.path.join(chap_dir, image_filename)

                with open(image_path, "wb") as f:
                    f.write(image_bytes)

    total_images += image_count

    print(
        f"Chapitre {chap_idx} "
        f"({os.path.basename(ppt_path)}) : "
        f"{image_count} image(s) extraite(s)"
    )

print(f"\nTotal : {total_images} image(s) extraite(s)")
print(f"Dossier de sortie : {output_dir}/")

Chapitre 1 (Cours_TS_Chap1.pptx) : 46 image(s) extraite(s)
Chapitre 2 (Cours_TS_Chap2.pptx) : 52 image(s) extraite(s)

Total : 98 image(s) extraite(s)
Dossier de sortie : ../data/extracted_images/


### Main : structuration des données et sauvegarde JSON
Finalement, on parcours les slides du ppt et on affecte à chacune son **id, section(titre), notions, catégories et contenu** en faisant appel au fonctions déjà définies.

In [13]:
from pptx import Presentation

# Reinitialisation du cours
course["chapters"]    = []
course["course_plan"] = [{"chap_id": 0, "chap_name": "Introduction"}]

# Boucle sur tous les fichiers .pptx
for chap_idx, ppt_path in enumerate(PPT_FILES, start=1):
    prs = Presentation(ppt_path)

    # Nom du chapitre extrait de la premiere slide
    first_slide_text = "\n".join(
        shape.text for shape in prs.slides[0].shapes if shape.has_text_frame
    )
    chap_name = extract_chapter_name(first_slide_text, ppt_path)

    chapter = {
        "chap_id"  : chap_idx,
        "chap_name": chap_name,
        "content"  : []
    }

    # Mise a jour du plan du cours
    course["course_plan"].append({"chap_id": chap_idx, "chap_name": chap_name})

    print(f"\n{'='*60}")
    print(f"Chapitre {chap_idx} : {chap_name}")
    print(f"Fichier  : {os.path.basename(ppt_path)}")
    print(f"{'='*60}")

    # Traitement des slides (on saute la slide de titre, index 0)
    for i, slide in enumerate(prs.slides):
        if i == 0:  # slide de titre -> deja utilisee pour chap_name
            continue

        slide_number = i + 1

        section    = extract_slide_section(slide)
        content    = extract_slide_content(slide, section, slide_number)
        notions    = extract_notions(chap_name, section, content)
        categories = extract_categories(slide, content, slide_number, chap_id=chap_idx)

        slide_dict = {
            "slide_id"     : slide_number,
            "section"      : section,
            "categories"   : categories,
            "notions"      : notions,
            "slide_content": content
        }
        chapter["content"].append(slide_dict)

    course["chapters"].append(chapter)
    print(f"  -> {len(chapter['content'])} slide(s) traite(s)")

# Resume final
print(f"\n{'='*60}")
print(f"Cours : {course['course_name']}")
print(f"Plan  : {len(course['course_plan'])} chapitre(s) (dont Introduction)")
print(f"{'='*60}")
pprint(course["course_plan"])



Chapitre 1 : Représentation temporelle des signaux
Fichier  : Cours_TS_Chap1.pptx
  -> 43 slide(s) traite(s)

Chapitre 2 : Systèmes linéaires continus
Fichier  : Cours_TS_Chap2.pptx
  -> 32 slide(s) traite(s)

Cours : Traitement du signal
Plan  : 3 chapitre(s) (dont Introduction)
[{'chap_id': 0, 'chap_name': 'Introduction'},
 {'chap_id': 1, 'chap_name': 'Représentation temporelle des signaux'},
 {'chap_id': 2, 'chap_name': 'Systèmes linéaires continus'}]


Pour terminer, on **sauvegarde** le résultat final en **JSON** :

In [14]:
import json
import os

# Créer le dossier de sortie si nécessaire
os.makedirs("../data/2-json", exist_ok=True)

JSON_PATH = "../data/2-json/cours.json"

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(course, f, indent=4, ensure_ascii=False, default=str)

print(f"JSON sauvegardé avec succès → {JSON_PATH}")


JSON sauvegardé avec succès → ../data/2-json/cours.json


Pour charger le JSON :

In [15]:
with open("../data/2-json/cours.json", "r", encoding="utf-8") as f:
    course_loaded = json.load(f)
#pprint(course_loaded)